In [7]:
import os
from copy import deepcopy
from MDToolkit.IO.read_file import pdb_file_to_structured_system
from MDToolkit.paths import PDB_FILES, CIF_FILES
from MDToolkit.custom_systems.membranes import cif_file_to_monolayer_membrane
from MDToolkit.utils.cutom_systems_utils import create_pore_circular
from MDToolkit.utils.structure_file_utils import read_molecular_data_json_entry
from MDToolkit.IO.write_file import write_lammps_structure_file_atomic_full
from MDToolkit.custom_systems.liquids import create_water_box

In [8]:
#####################################################
# CNT Graphene Water System
#####################################################

# Import CNT
CNT_file_path = os.path.join(PDB_FILES, "CNT_12_12_10.pdb")
CNT_system = pdb_file_to_structured_system(CNT_file_path)
CNT_system.rotate_system_spherical(90, 0, 0)
CNT_system.combine_all_molecules()
CNT_system.set_COM_to_origin()

# Generate Graphene Layers 
graphene_cif_file_path = os.path.join(CIF_FILES, "graphene.cif")
membrane_dims = [52, 52, 0]
monolayer_graphene_system = cif_file_to_monolayer_membrane(graphene_cif_file_path, max_dimension = membrane_dims, shrink_buffer=[2.0, 0.71, 0.71])
monolayer_graphene_system.set_COM_to_origin()
monolayer_graphene_system, pore_area = create_pore_circular(monolayer_graphene_system, radius = 7.5)
monolayer_graphene_system.combine_all_molecules()

monolayer_graphene_system_copy = deepcopy(monolayer_graphene_system)

# Combine CNT and Graphenes
monolayer_graphene_system.set_COM_to_origin([-52, 0, 0])
CNT_system.combine_with_other_structured_system(monolayer_graphene_system)
monolayer_graphene_system_copy.set_COM_to_origin([52, 0, 0])
CNT_system.combine_with_other_structured_system(monolayer_graphene_system_copy)
CNT_system.combine_all_molecules()

# Create Water Boxes
water_box_bounds_1 = {
  "min_x" : -104, "max_x" : -54,
  "min_y" : -25.0, "max_y" : 25.0,
  "min_z" : -25.0, "max_z" : 25.0
}
water_box_bounds_2 = {
  "min_x" : 54, "max_x" : 104,
  "min_y" : -25.0, "max_y" : 25.0,
  "min_z" : -25.0, "max_z" : 25.0
}
water_box_1 = create_water_box(water_box_bounds_1, "SPCE", density_correction=0.1)
water_box_2 = create_water_box(water_box_bounds_2, "SPCE", density_correction=0.1)
CNT_system.combine_with_other_structured_system(water_box_1)
CNT_system.combine_with_other_structured_system(water_box_2)
CNT_system.populate_elemental_properties_for_all_atoms()

for i in range(1, len(CNT_system.molecule_list)):
  CNT_system.molecule_list[i].populate_bonds_from_molecular_data(read_molecular_data_json_entry(alias_key = "H2O"))
  CNT_system.molecule_list[i].populate_angles_from_molecular_data(read_molecular_data_json_entry(alias_key = "H2O"))

# Write Structure

write_lammps_structure_file_atomic_full(CNT_system, file_name = "CNT_Graphene_Water.data")

H2O Template File: /workspace/code/MDToolkit/MDToolkit/data/pdb_files/H2O_SPCE.pdb
Packmol input file successfully written to /workspace/code/MDToolkit/packmol_input_files/H2O_box_packmol_helper.inp

################################################################################

 PACKMOL - Packing optimization for the automated generation of
 starting configurations for molecular dynamics simulations.
 
                                                             Version 21.2.3 

################################################################################

  Packmol must be run with: packmol < inputfile.inp 

  Userguide at: http://m3g.iqm.unicamp.br/packmol 

  Reading input file... (Control-C aborts)
  Types of coordinate files specified: pdb
  Seed for random number generator:        81033
  Output file: /workspace/code/MDToolkit/Output/H2O_box.pdb
  Reading coordinate file: /workspace/code/MDToolkit/MDToolkit/data/pdb_files/H2O_SPCE.pdb
  Number of independent structures:    